# H-010 · Score × confidence sizing

Arms: equal risk vs `|score|` vs `|score|*(1-p_ADF)`. Fold-train mean abs score rescales to ~1.

Type `SIZE_STAR`.


## 0. Imports & Config


In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from backtest.s2_coint.report import (
    fold_table,
    fold_val_metrics,
    load_star_stack,
    median_sharpe_hint,
    plot_fold_boxplots,
    require_star,
    save_star_stack,
    write_tearsheet_pdf,
)
from backtest.s2_coint.research import (
    ARTIFACTS_DIR,
    DEFAULT_STAR_STACK,
    config_from_stack,
    is_end_for_stack,
    load_s1_weekly,
    load_universe_c_panels,
    lookbacks_for_bar,
    overlay_kalman_hedge,
    repo_root,
    split_is_oos,
    tearsheet_path,
)
from backtest.s2_coint.runner import run_s2_backtest
from backtest.s2_coint.walkforward import embargo_bars_for_config, make_s2_folds
from strategies.s2_coint.config import S2SimConfig

STAR_PATH = DEFAULT_STAR_STACK
TEARSHEET_DIR = ARTIFACTS_DIR
stack = load_star_stack(STAR_PATH)
PAIRS_STAR = list(stack["PAIRS_STAR"])
print("stack keys:", sorted(stack))
print("PAIRS_STAR", PAIRS_STAR)


## 1. Load frozen panel / PAIRS_STAR


In [ ]:
require_star("BAR_STAR", stack.get("BAR_STAR"))
BAR = str(stack["BAR_STAR"])
train, full = load_universe_c_panels(BAR, PAIRS_STAR, root=ROOT)
if stack.get("HEDGE_STAR") == "kalman":
    lb = lookbacks_for_bar(BAR)
    train = overlay_kalman_hedge(train, burn_in=lb["kalman_burn_in"], z_window=lb["z_window"], hl_window=lb["hl_window"])
    full = overlay_kalman_hedge(full, burn_in=lb["kalman_burn_in"], z_window=lb["z_window"], hl_window=lb["hl_window"])
is_end = is_end_for_stack(stack, full if BAR == "1h" else train)
if BAR == "1d":
    is_panel, oos_panel = train.copy(), full.loc[pd.to_datetime(full["date"]) > is_end].copy()
else:
    is_panel, oos_panel = split_is_oos(full, is_end=is_end)
s1_weekly = load_s1_weekly(ROOT)
print("bar", BAR, "is_end", is_end, "IS rows", len(is_panel), "OOS rows", len(oos_panel))
is_panel.head()


## 2. Attach this-hyp columns


In [ ]:
panel = is_panel.copy()
print(panel.columns.tolist())
panel.head()


## 3. Walk-forward folds


In [ ]:
dates = pd.DatetimeIndex(pd.to_datetime(panel["date"])).sort_values().unique()
folds = make_s2_folds(dates, n_folds=3, embargo_bars=embargo_bars_for_config(bar=BAR))
fold_table(folds)


## 4. Fold-val metrics (validation only)


In [ ]:
base = config_from_stack(stack)
configs = {
    "equal": config_from_stack(stack, size_mode="equal"),
    "score": config_from_stack(stack, size_mode="score"),
    "score_conf": config_from_stack(stack, size_mode="score_conf"),
}
fold_df = fold_val_metrics(panel, folds, configs, s1_weekly=s1_weekly)
fold_df


## 5. Boxplots (do not assign STAR here)


In [ ]:
plot_fold_boxplots(fold_df, title="H-010 fold-val")
plt.show()
print("median-Sharpe hint (commentary only):", median_sharpe_hint(fold_df))
fold_df.groupby("arm")[["ann_sharpe", "max_drawdown", "corr_to_s1"]].median()


## 6. Type `SIZE_STAR` then save


In [ ]:
SIZE_STAR = None  # TODO set after review — do not use argmax / median Sharpe
require_star("SIZE_STAR", SIZE_STAR)
stack["SIZE_STAR"] = SIZE_STAR
save_star_stack(STAR_PATH, stack)
print("wrote", STAR_PATH)


## 7. Sealed OOS once


In [ ]:
require_star("SIZE_STAR", SIZE_STAR)
oos_cfg = config_from_stack(load_star_stack(STAR_PATH))
oos = run_s2_backtest(oos_panel, oos_cfg, s1_weekly=s1_weekly)
print(oos.metrics)
write_tearsheet_pdf(tearsheet_path("H-010", str(SIZE_STAR)), oos.returns, title="H-010 sealed OOS")
print("tearsheet", tearsheet_path("H-010", str(SIZE_STAR)))


## 8. Notes for next hyp


H-011 vol-aware k_t vs S1 vol targeting is next.
